# Scaling Engine

In [ ]:
# Path setup: notebooks live in notebooks/, but every path below is relative to the repo root
import os
from pathlib import Path

_here = Path.cwd()
for _cand in (_here, *_here.parents):
    if (_cand / "README.md").exists() and (_cand / "notebooks").is_dir():
        os.chdir(_cand)
        break
print("Working directory:", Path.cwd())

In [ ]:
# Cell 1 – Setup, install hints, imports

import cv2
import cv2.aruco as aruco
import numpy as np

from dataclasses import dataclass, field
from typing import Dict, Any, Optional, List, Tuple

In [2]:
# Cell 2 – Core dataclasses and config

@dataclass
class ScaleResult:
    pixel_per_cm: float                # pixels for 1 cm
    tier: int                          # 1..5
    method: str                        # 'aruco', 'a4', 'ruler30', 'card', 'manual_two_point', 'distance_calibrated', 'carryover'
    confidence: float                  # 0.0–1.0
    extra: Dict[str, Any] = field(default_factory=dict)


@dataclass
class RefDetection:
    kind: str                          # 'a4', 'ruler30', 'card'
    box: np.ndarray                    # (4,2) polygon or (4,) bbox [x1,y1,x2,y2]


@dataclass
class DistanceCalibration:
    z_calibration_m: float             # distance used for calibration (m)
    pixel_per_cm_cal: float            # scale at that distance (px/cm)


REFERENCE_SIZES_CM = {
    "a4":      {"width": 21.0,  "height": 29.7},
    "card":    {"width": 8.56,  "height": 5.398},  # ISO/ID-1
    "ruler30": {"length": 30.0},
}


In [3]:
# Cell 3 – Tier 1: ArUco marker (generator + scale_from_aruco)

def generate_aruco_marker(
    marker_id: int = 0,
    marker_size_cm: float = 20.0,
    dpi: int = 300,
    dict_name: int = aruco.DICT_5X5_100,
    out_path: str = "assets/aruco/aruco_20cm_id0.png",
):
    """
    Generate a printable ArUco marker PNG of given physical size.
    marker_size_cm is the side length of the black square.
    """
    dictionary = aruco.getPredefinedDictionary(dict_name)
    marker_size_px = int(marker_size_cm / 2.54 * dpi)  # cm -> inch -> px

    # Handle different OpenCV versions
    if hasattr(aruco, "drawMarker"):
        img = aruco.drawMarker(dictionary, marker_id, marker_size_px)
    elif hasattr(aruco, "generateImageMarker"):
        img = aruco.generateImageMarker(dictionary, marker_id, marker_size_px)
    else:
        raise AttributeError("Your OpenCV build has no ArUco marker generator. Install opencv-contrib-python.")

    cv2.imwrite(out_path, img)
    print(f"Saved marker ({marker_size_cm} cm) to {out_path}")


def scale_from_aruco(
    bgr_img: np.ndarray,
    marker_size_cm: float = 20.0,
    dict_name: int = aruco.DICT_5X5_100,
) -> Optional[ScaleResult]:
    """
    Tier 1 – Detect an ArUco marker and compute pixels-per-cm from its side length.
    Returns None if no marker found.
    """
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    dictionary = aruco.getPredefinedDictionary(dict_name)
    parameters = aruco.DetectorParameters()
    detector = aruco.ArucoDetector(dictionary, parameters)
    corners, ids, _ = detector.detectMarkers(gray)

    if ids is None or len(ids) == 0:
        return None

    # Use the largest detected marker (closest, best resolution)
    areas = []
    for c in corners:
        pts = c[0].astype(np.float32)
        areas.append(cv2.contourArea(pts))
    idx = int(np.argmax(areas))
    pts = corners[idx][0].astype(np.float32)  # (4,2)

    side_lengths = [
        np.linalg.norm(pts[0] - pts[1]),
        np.linalg.norm(pts[1] - pts[2]),
        np.linalg.norm(pts[2] - pts[3]),
        np.linalg.norm(pts[3] - pts[0]),
    ]
    avg_side_px = float(np.mean(side_lengths))

    pixel_per_cm = avg_side_px / marker_size_cm

    return ScaleResult(
        pixel_per_cm=pixel_per_cm,
        tier=1,
        method="aruco",
        confidence=0.95,
        extra={
            "marker_side_px": avg_side_px,
            "marker_size_cm": marker_size_cm,
            "num_markers": len(corners),
        },
    )


In [4]:
generate_aruco_marker()  # Generate default marker on import

Saved marker (20.0 cm) to aruco_20cm_id0.png


In [5]:
# Cell 4 – Tier 2: Known reference objects (A4, 30 cm ruler, credit card)

def _bbox_from_box(box: np.ndarray) -> Tuple[float, float, float, float]:
    """
    Ensure we have a simple [x1,y1,x2,y2] bbox from input box.
    Accepts either (4,2) polygon or (4,) bbox.
    """
    box = np.asarray(box, dtype=np.float32)
    if box.shape == (4, 2):
        xs = box[:, 0]
        ys = box[:, 1]
        return float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())
    elif box.shape == (4,):
        x1, y1, x2, y2 = box
        return float(x1), float(y1), float(x2), float(y2)
    else:
        raise ValueError(f"Unsupported box shape: {box.shape}")


def scale_from_reference_objects(
    bgr_img: np.ndarray,
    detections: List[RefDetection],
) -> Optional[ScaleResult]:
    """
    Tier 2 – Use known-size objects (A4, ruler, card) to compute pixels-per-cm.
    You must provide detections (e.g. from a detector model or contour logic).
    """
    if not detections:
        return None

    best: Optional[ScaleResult] = None

    for det in detections:
        kind = det.kind
        sz = REFERENCE_SIZES_CM.get(kind)
        if sz is None:
            continue

        x1, y1, x2, y2 = _bbox_from_box(det.box)
        width_px = abs(x2 - x1)
        height_px = abs(y2 - y1)

        if kind == "a4":
            # choose side that is more visible in image
            if width_px >= height_px:
                pixel_per_cm = width_px / sz["width"]
                used_dim = ("width", sz["width"], width_px)
            else:
                pixel_per_cm = height_px / sz["height"]
                used_dim = ("height", sz["height"], height_px)

        elif kind == "card":
            pixel_per_cm = width_px / sz["width"]
            used_dim = ("width", sz["width"], width_px)

        elif kind == "ruler30":
            long_px = max(width_px, height_px)
            pixel_per_cm = long_px / sz["length"]
            used_dim = ("length", sz["length"], long_px)

        else:
            continue

        cand = ScaleResult(
            pixel_per_cm=pixel_per_cm,
            tier=2,
            method=kind,
            confidence=0.85,
            extra={
                "used_dim": used_dim,
                "bbox": (x1, y1, x2, y2),
            },
        )

        # Keep detection with largest pixel coverage (more reliable)
        if best is None or (width_px * height_px) > best.extra.get("area_px", 0):
            cand.extra["area_px"] = width_px * height_px
            best = cand

    return best


In [6]:
# Cell 5 – Tier 3: Manual two-point calibration (user taps two points)

def scale_from_manual_two_points(
    pt1_norm: Tuple[float, float],
    pt2_norm: Tuple[float, float],
    real_length_cm: float,
    img_shape: Tuple[int, int, int],
) -> ScaleResult:
    """
    Tier 3 – User taps two points and enters real distance (cm).
    pt1_norm, pt2_norm: (x,y) in [0,1] normalized coords from frontend.
    img_shape: (H, W, C).
    """
    H, W = img_shape[:2]
    x1, y1 = pt1_norm[0] * W, pt1_norm[1] * H
    x2, y2 = pt2_norm[0] * W, pt2_norm[1] * H

    d_px = float(np.hypot(x2 - x1, y2 - y1))
    pixel_per_cm = d_px / real_length_cm

    return ScaleResult(
        pixel_per_cm=pixel_per_cm,
        tier=3,
        method="manual_two_point",
        confidence=0.8,
        extra={
            "pt1_px": (x1, y1),
            "pt2_px": (x2, y2),
            "d_px": d_px,
            "real_length_cm": real_length_cm,
        },
    )


In [7]:
# Cell 6 – Tier 4: Distance + device calibration (pinhole-style)

def scale_from_distance_calibrated(
    distance_m: float,
    calib: DistanceCalibration,
) -> ScaleResult:
    """
    Tier 4 – User enters camera→cow distance, we rescale using a stored calibration.
    Assumes pixel_per_cm ∝ 1 / distance (pinhole camera model).
    """
    z_cal = calib.z_calibration_m
    # Scale is inversely proportional to distance in pinhole model
    pixel_per_cm_new = calib.pixel_per_cm_cal * (z_cal / distance_m)

    return ScaleResult(
        pixel_per_cm=pixel_per_cm_new,
        tier=4,
        method="distance_calibrated",
        confidence=0.7,  # depends heavily on user distance accuracy
        extra={
            "z_calibration_m": z_cal,
            "pixel_per_cm_cal": calib.pixel_per_cm_cal,
            "distance_m": distance_m,
        },
    )


In [8]:
# Cell 7 – Tier 5: Carryover and orchestrator (estimate_scale)

def scale_from_carryover(prev_scale: ScaleResult) -> ScaleResult:
    """
    Tier 5 – Reuse previous scale from same session if camera unchanged.
    """
    return ScaleResult(
        pixel_per_cm=prev_scale.pixel_per_cm,
        tier=5,
        method="carryover",
        confidence=prev_scale.confidence * 0.9,
        extra={"base_method": prev_scale.method},
    )


def estimate_scale(
    bgr_img: np.ndarray,
    ref_detections: Optional[List[RefDetection]] = None,
    manual_two_point: Optional[Tuple[Tuple[float, float], Tuple[float, float], float]] = None,
    distance_m: Optional[float] = None,
    device_calib: Optional[DistanceCalibration] = None,
    prev_scale: Optional[ScaleResult] = None,
) -> Optional[ScaleResult]:
    """
    Try tiers in priority order: 1 (aruco) → 2 (known objects) → 3 (manual two-point)
    → 4 (distance + calibration) → 5 (carryover).
    Returns best ScaleResult or None.
    """
    # Tier 1 – ArUco
    s = scale_from_aruco(bgr_img)
    if s is not None:
        return s

    # Tier 2 – reference objects
    if ref_detections:
        s = scale_from_reference_objects(bgr_img, ref_detections)
        if s is not None:
            return s

    # Tier 3 – manual two-point
    if manual_two_point is not None:
        pt1, pt2, real_cm = manual_two_point
        H, W = bgr_img.shape[:2]
        return scale_from_manual_two_points(pt1, pt2, real_cm, (H, W, 3))

    # Tier 4 – distance-calibrated
    if distance_m is not None and device_calib is not None:
        return scale_from_distance_calibrated(distance_m, device_calib)

    # Tier 5 – carryover
    if prev_scale is not None:
        return scale_from_carryover(prev_scale)

    # No scale available
    return None


In [9]:
# Cell 8 – Example usage on one image (optional demo)

img_path = r"vetra_side_pose_dataset\images\train\2_c3.jpg"
bgr = cv2.imread(img_path)

# Example: try only ArUco + manual fallback
scale_res = estimate_scale(
    bgr_img=bgr,
    ref_detections=None,
    manual_two_point=None,  # e.g. ((0.1,0.8), (0.3,0.8), 30.0)
    distance_m=None,
    device_calib=None,
    prev_scale=None,
)

print(scale_res)


None
